# Step 2 — Pilot 标签与数据质量

用数学 grader 写入 `correct`，检查完成率、正确率、正负样本平衡和长度关系。

**Go/No-Go：** 正确率最好落在 30%–70%；至少要求两类都有、少数类 ≥10%。若不满足，应调整题目难度，而不是直接扩采。

In [ ]:
# @title Step 02.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow','tqdm']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

CONFIG = load_config(REPO)
print("Experiment:", CONFIG["experiment_name"])


In [ ]:
# @title Step 02.2 — 运行 pilot grader
RUN_STAGE = True
pilot_path = REPO / CONFIG["paths"]["pilot"]
grader_metrics = REPO / "artifacts/metrics/pilot_grader.json"
if RUN_STAGE:
    run_repo(
        REPO,
        "python3", "src/evaluate_and_label_rollouts.py",
        "--data", pilot_path,
        "--model", CONFIG["grader_model_id"],
        "--tensor-parallel-size", 1,
        "--gpu-memory-utilization", CONFIG["gpu_memory_utilization"],
        "--max-model-len", CONFIG["grader_max_model_len"],
        "--max-num-seqs", CONFIG["max_num_seqs"],
        "--dtype", CONFIG["dtype"],
        "--output-json", grader_metrics,
        "--show-examples",
    )

In [ ]:
# @title Step 02.3 — 可视化质量并作出 Go / No-Go
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

df = pd.read_parquet(pilot_path)
missing = require_columns(df, ["finished", "correct", "length"])
if missing:
    raise ValueError(f"缺少列: {missing}")
accuracy = float(df["correct"].mean())
class_share = df["correct"].value_counts(normalize=True)
minority_share = float(class_share.min()) if len(class_share) == 2 else 0.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pd.crosstab(df["finished"], df["correct"]).plot.bar(stacked=True, ax=axes[0], color=["#e45756", "#49beaa"])
axes[0].set_title("Finished × Correct")
df["correct"].value_counts().sort_index().plot.bar(ax=axes[1], color=["#e45756", "#49beaa"])
axes[1].set_title(f"Correctness balance ({accuracy:.1%} correct)")
for label, group in df.groupby("correct"):
    axes[2].hist(group["length"], bins=25, alpha=.55, label=f"correct={label}")
axes[2].set(title="Length by correctness", xlabel="output tokens")
axes[2].legend()
plt.tight_layout()
plt.show()

display(df.groupby("correct")["length"].describe().round(1))
checks = [
    gate("Correct 标签完整", not missing and df["correct"].notna().all(), f"null={int(df['correct'].isna().sum())}"),
    gate("正负样本同时存在", df["correct"].nunique() == 2, str(df["correct"].value_counts().to_dict()), kind="scientific"),
    gate("少数类 ≥10%", minority_share >= 0.10, f"minority={minority_share:.1%}", kind="scientific"),
    gate("理想正确率 30%–70%", 0.30 <= accuracy <= 0.70, f"accuracy={accuracy:.1%}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "02_pilot_quality", checks, {"accuracy": accuracy, "minority_share": minority_share, "finished_rate": float(df['finished'].mean())})